In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/datasets/100_Round_3 - Final Annotations.csv"  # queries
B_PATH = "/home/ubuntu/datasets/400_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/r4_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/r4_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/similarity_scores/cross_similar_posts_k3_100_r4.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['body'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id subreddit                                              title  \
 0  1ldqw1t  abortion  i am pregnant and i don’t know if i should get...   
 1  1ldoh06  abortion                               abortion not worked?   
 2  1ldmq62  abortion   Taking miso today, terrified of the blood & pain   
 3  1ldlmrv  abortion  Struggling to hide my pregnancy from family......   
 4  1ldl2yt  abortion  Bleeding 2 weeks exactly today after abortion ...   
 
                                                 body          created_utc  \
 0  i just found out that i (20f) am pregnant with...  2025-06-17 15:58:02   
 1  hi! 24 hours, no blood. AT ALL. after taking t...  2025-06-17 14:23:49   
 2  I took mifepristone yesterday and will be taki...  2025-06-17 13:10:20   
 3  For some context, I (27) live with my parents,...  2025-06-17 12:18:46   
 4  I had a SA and I was having light back/brown-r...  2025-06-17 11:51:29   
 
                                                  url  \
 0  https://www

In [2]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
r1_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r1_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
r1_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r1_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [3]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [4]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(r1_emb_A, r1_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/similarity_scores/cross_similar_posts_k3_100_r3.json
